# <font color="#418FDE" size="6.5" uppercase>**Sequenzmodelle**</font>

>Last update: 20260723.
    
By the end of this Lecture, you will be able to:
- Bereiten Sequenzfenster und Baselines für zeitliche Vorhersage- oder Klassifikationsaufgaben vor. 
- Trainieren kleine Conv1D-, SimpleRNN- und LSTM- oder GRU-Modelle CPU-freundlich. 
- Vergleichen Sequenznetze mit klassischen Signalmodellen anhand zeitlicher Fehleranalysen. 


## **1. Sequenzen vorbereiten**

### **1.1. Batchformen verstehen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_01.jpg?v=1784816933" width="250">



>* Sequenzen werden in gleichförmige Fenster gestapelt
>* Batchform bestimmt sichtbare zeitliche Zusammenhänge

>* Fensterlänge bestimmt sichtbare zeitliche Muster
>* Dimensionen müssen zur Modellarchitektur passen

>* Keine Zukunftsdaten ins Eingabefenster lassen
>* Labels und Zeitlogik klar festlegen



In [ ]:
#@title Python-Code - Batchformen verstehen

# Wir formen eine Messreihe zu Sequenzfenstern.
# Batchformen zeigen Beispiele, Zeit und Merkmale.
# Die Ausgabe vergleicht korrekte und falsche Formen.

import numpy as np

# Eine kleine Messreihe enthält zwei Merkmale pro Zeitpunkt.
time_steps = np.arange(10)
temperature = 20 + 0.5 * time_steps
humidity = 60 - 1.0 * time_steps

# Spalten sind Merkmale, Zeilen sind zeitlich geordnet.
series = np.column_stack((temperature, humidity))
window_length = 4

# Jedes Fenster nutzt nur vergangene Werte als Eingabe.
windows = []
targets = []
for start in range(len(series) - window_length):
    end = start + window_length
    windows.append(series[start:end])
    targets.append(series[end, 0])

# Aus Listen entsteht die typische Batchform.
X = np.array(windows)
y = np.array(targets)

# Eine einfache Prüfung macht die Bedeutung der Achsen sichtbar.
expected_shape = (6, 4, 2)
if X.shape != expected_shape:
    raise ValueError("Die Fensterform passt nicht zur erwarteten Batchform.")

# Diese Umformung wäre technisch möglich, aber fachlich falsch.
wrong_shape = X.reshape(X.shape[0], X.shape[2], X.shape[1])

print("Batchform X: Beispiele x Zeitschritte x Merkmale =", X.shape)
print("Zielform y: ein Zielwert pro Beispiel =", y.shape)
print("Erstes Fenster, Temperaturwerte:", np.round(X[0, :, 0], 1).tolist())
print("Erstes Fenster, Feuchtewerte:", np.round(X[0, :, 1], 1).tolist())
print("Ziel nach dem ersten Fenster:", round(float(y[0]), 1))
print("Falsche Achsenreihenfolge hätte Form:", wrong_shape.shape)



### **1.2. Naive Vergleichsmodelle**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_02.jpg?v=1784816935" width="250">



>* Einfache Baselines setzen Mindestmaßstäbe für Sequenzmodelle
>* Letzte Werte oder Klassen oft überraschend stark

>* Baseline passend zu Aufgabe und Daten wählen
>* Trainierte Modelle müssen einfache Regeln übertreffen

>* Baselines fair mit gleichen Zeitfenstern vergleichen
>* Fehler zeitlich prüfen und Lecks erkennen



In [ ]:
#@title Python-Code - Naive Vergleichsmodelle

# Dieses Beispiel baut einfache Sequenzfenster.
# Naive Baselines nutzen nur vergangene Werte.
# Der Plot vergleicht Zielwerte und Vorhersagen.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kleine, glatte Zeitreihe mit Rauschen.
rng = np.random.default_rng(42)
time_steps = np.arange(80)
trend = 0.04 * time_steps
season = 2.0 * np.sin(time_steps / 6.0)
noise = rng.normal(0.0, 0.25, size=time_steps.size)
values = 20.0 + trend + season + noise

# Jedes Fenster enthält nur vergangene Beobachtungen.
window_size = 8
X = []
y = []
for start in range(len(values) - window_size):
    X.append(values[start:start + window_size])
    y.append(values[start + window_size])

# Listen werden zu Arrays für einfache Berechnungen.
X = np.array(X)
y = np.array(y)
if X.shape[0] != y.shape[0]:
    raise ValueError("Fenster und Zielwerte passen nicht zusammen.")

# Ein zeitlicher Split verhindert Zukunftsinformation im Training.
split_index = int(0.7 * len(y))
X_test = X[split_index:]
y_test = y[split_index:]
test_time = time_steps[window_size + split_index:]

# Zwei naive Regeln verwenden nur das jeweilige Eingabefenster.
last_value_pred = X_test[:, -1]
window_mean_pred = X_test.mean(axis=1)

# Der mittlere absolute Fehler macht Baselines vergleichbar.
last_mae = np.mean(np.abs(y_test - last_value_pred))
mean_mae = np.mean(np.abs(y_test - window_mean_pred))
print(f"Testfenster: {len(y_test)}")
print(f"MAE letzter Wert: {last_mae:.2f} Grad Celsius")
print(f"MAE Fenstermittel: {mean_mae:.2f} Grad Celsius")

# Der Plot zeigt, wann welche naive Regel näher liegt.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(test_time, y_test, label="echter nächster Wert", linewidth=2)
ax.plot(test_time, last_value_pred, label="Baseline: letzter Wert")
ax.plot(test_time, window_mean_pred, label="Baseline: Fenstermittel")
ax.set_title("Naive Vergleichsmodelle für Ein-Schritt-Vorhersage")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("Temperatur in Grad Celsius")
ax.legend()
plt.show()



### **1.3. Zeitliche Splits**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_01_03.jpg?v=1784816937" width="250">



>* Zufälliges Mischen verursacht Zukunftslecks
>* Zeitliche Splits testen echte Vorhersagen

>* Erst zeitlich splitten, dann Fenster bilden
>* Puffer verhindern Leckagen zwischen Teilmengen

>* Split an Einsatz und Zukunftshorizont anpassen
>* Mehrere Zeitabschnitte zeigen stabile Modellleistung



In [ ]:
#@title Python-Code - Zeitliche Splits

# Dieses Beispiel zeigt saubere zeitliche Datensplits.
# Fenster entstehen erst nach der zeitlichen Trennung.
# Die Ausgabe vergleicht korrekte und riskante Aufteilung.

import numpy as np
import matplotlib.pyplot as plt

# Eine kleine künstliche Zeitreihe simuliert stündliche Messwerte.
rng = np.random.default_rng(42)
time_steps = np.arange(80)

trend = 0.03 * time_steps
season = np.sin(time_steps / 4)
noise = rng.normal(0, 0.12, size=time_steps.size)
series = trend + season + noise

# Diese Funktion erzeugt Eingabefenster und den nächsten Zielwert.
def make_windows(values, window_size):
    inputs = []
    targets = []

    for start in range(len(values) - window_size):
        end = start + window_size
        inputs.append(values[start:end])
        targets.append(values[end])

    return np.array(inputs), np.array(targets)

window_size = 6
train_end = 50
validation_end = 65

if validation_end + window_size > len(series):
    raise ValueError("Die Zeitreihe ist für dieses Beispiel zu kurz.")

# Korrekt: Erst Zeitbereiche trennen, dann Fenster bilden.
train_values = series[:train_end]
validation_values = series[train_end:validation_end]
test_values = series[validation_end:]

X_train, y_train = make_windows(train_values, window_size)
X_validation, y_validation = make_windows(validation_values, window_size)
X_test, y_test = make_windows(test_values, window_size)

# Riskant: Erst Fenster bilden, dann zufällig mischen.
all_windows, all_targets = make_windows(series, window_size)
random_order = rng.permutation(len(all_windows))

random_train_count = len(X_train)
random_validation_count = len(X_validation)
random_train_indices = random_order[:random_train_count]
random_validation_indices = random_order[
    random_train_count:random_train_count + random_validation_count
]

# Diese Startpunkte zeigen, ob spätere Zeitpunkte im Training landen.
random_train_starts = random_train_indices
latest_random_train_start = int(random_train_starts.max())

print("Zeitlicher Split: Training, Validierung und Test bleiben geordnet.")
print(f"Trainingsfenster: {len(X_train)}, Validierungsfenster: {len(X_validation)}")
print(f"Testfenster: {len(X_test)}, Fensterlänge: {window_size}")
print(f"Riskanter Zufallssplit nutzt Trainingsfenster bis Startindex {latest_random_train_start}.")
print("Das kann Zukunftsinformation in die Bewertung bringen.")

# Die Grafik markiert zusammenhängende Zeitbereiche vor der Fensterbildung.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, series, color="black", linewidth=1.5, label="Zeitreihe")
ax.axvspan(0, train_end - 1, color="tab:blue", alpha=0.18, label="Training")
ax.axvspan(train_end, validation_end - 1, color="tab:orange", alpha=0.22, label="Validierung")
ax.axvspan(validation_end, len(series) - 1, color="tab:green", alpha=0.18, label="Test")

ax.set_title("Zeitliche Splits vor der Fensterbildung")
ax.set_xlabel("Zeitindex")
ax.set_ylabel("Messwert")
ax.legend(loc="upper left")
plt.show()



## **2. Kleine Sequenznetze**

### **2.1. Conv1D für Sequenzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_01.jpg?v=1784816941" width="250">



>* Conv1D erkennt lokale Muster in Sequenzen
>* Effizient und gut für CPU-Experimente

>* Kleine Conv1D-Netze gezielt schlank halten
>* Filter erkennen lokale Muster schnell

>* Conv1D trainiert effizient und ressourcenschonend
>* Begrenzter Kontext erfordert Fehleranalyse



In [ ]:
#@title Python-Code - Conv1D für Sequenzen

# Wir bauen ein kleines Conv1D-ähnliches Sequenzmodell.
# Lokale Filter erkennen kurze Muster in Zeitfenstern.
# Die Grafik zeigt gelernte Vorhersagen und Fehler.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import train_test_split
import sklearn

# Eine deterministische Zeitreihe simuliert lokale Wellenmuster.
rng = np.random.default_rng(42)
time = np.arange(360)

signal = np.sin(time / 9) + 0.35 * np.sin(time / 3)
noise = rng.normal(0, 0.12, size=time.size)
series = signal + noise

# Jedes Trainingsbeispiel ist ein kurzes Sequenzfenster.
window_size = 18
X_windows = []
y_next = []

for start in range(len(series) - window_size):
    end = start + window_size
    X_windows.append(series[start:end])
    y_next.append(series[end])

X_windows = np.array(X_windows)
y_next = np.array(y_next)

# Die Form entspricht Samples, Zeitschritten und Kanälen.
X_sequence = X_windows[:, :, np.newaxis]
expected_shape = (len(y_next), window_size, 1)

if X_sequence.shape != expected_shape:
    raise ValueError("Die Sequenzfenster haben eine unerwartete Form.")

# Kleine lokale Filter imitieren eine Conv1D-Merkmalsschicht.
filters = np.array([
    [1.0, 1.0, 1.0],
    [-1.0, 0.0, 1.0],
    [1.0, -2.0, 1.0],
])

features = []
for window in X_windows:
    row = []
    for kernel in filters:
        conv_values = np.convolve(window, kernel[::-1], mode="valid")
        row.append(np.mean(conv_values))
        row.append(np.max(conv_values))
    features.append(row)

features = np.array(features)

# Ein kleines lineares Modell nutzt die lokalen Conv1D-Merkmale.
X_train, X_test, y_train, y_test = train_test_split(
    features, y_next, test_size=0.25, random_state=42, shuffle=False
)

model = Ridge(alpha=1.0)
model.fit(X_train, y_train)
predictions = model.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
baseline = mean_absolute_error(y_test[1:], y_test[:-1])

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Fensterform wie bei Conv1D: {X_sequence.shape}")
print(f"MAE Conv1D-Merkmale: {mae:.3f}")
print(f"MAE naive Baseline: {baseline:.3f}")

# Die Kurven zeigen Vorhersagequalität über die Zeit.
plot_steps = np.arange(len(y_test))[:90]
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(plot_steps, y_test[:90], label="Echter nächster Wert")
ax.plot(plot_steps, predictions[:90], label="Vorhersage")
ax.set_title("Lokale Conv1D-Merkmale für Sequenzvorhersage")
ax.set_xlabel("Test-Zeitschritt")
ax.set_ylabel("Signalwert")
ax.legend()
plt.show()



### **2.2. SimpleRNN Grundlagen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_02.jpg?v=1784816939" width="250">



>* Verarbeitet Sequenzen Schritt für Schritt
>* Nutzt Zustand als kompaktes Gedächtnis

>* SimpleRNN trainiert schnell bei kurzen Sequenzen
>* Lange Abhängigkeiten überfordern einfache Zustände

>* Fehler zeitlich statt nur insgesamt prüfen
>* SimpleRNN klein halten und mit LSTM vergleichen



In [ ]:
#@title Python-Code - SimpleRNN Grundlagen

# Dieses Beispiel zeigt ein kleines SimpleRNN-Prinzip.
# Ein Zustand speichert Informationen aus früheren Zeitschritten.
# Die Grafik vergleicht Signal, Zustand und Vorhersage.

import numpy as np
import matplotlib.pyplot as plt

# Wir erzeugen eine kurze, deterministische Zeitreihe.
rng = np.random.default_rng(42)
time_steps = np.arange(60)
trend = 0.03 * time_steps
season = np.sin(time_steps / 4)
noise = rng.normal(0, 0.12, size=time_steps.size)
signal = trend + season + noise

# Ein Sequenzfenster enthält mehrere aufeinanderfolgende Messwerte.
window_size = 8
if signal.size <= window_size:
    raise ValueError("Die Zeitreihe ist zu kurz für dieses Fenster.")

# Diese Gewichte sind klein und bleiben hier bewusst fest.
input_weight = 0.55
state_weight = 0.75
output_weight = 1.05

# Der rekurrente Zustand wird Schritt für Schritt aktualisiert.
states = []
state = 0.0
for value in signal:
    state = np.tanh(input_weight * value + state_weight * state)
    states.append(state)

states = np.array(states)
prediction = output_weight * states

# Wir bewerten nur die nächsten Werte nach dem ersten Fenster.
valid_signal = signal[window_size:]
valid_prediction = prediction[window_size - 1:-1]
mae = np.mean(np.abs(valid_signal - valid_prediction))

print("SimpleRNN-Idee ohne Deep-Learning-Bibliothek")
print(f"Fensterlänge: {window_size} Zeitschritte")
print(f"Mittlerer absoluter Fehler: {mae:.3f}")
print("Der Zustand glättet frühere Werte und beeinflusst die nächste Schätzung.")

# Die Grafik macht den zeitlichen Zustand sichtbar.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, signal, label="echtes Signal", linewidth=2)
ax.plot(time_steps, states, label="rekurrenter Zustand", linewidth=2)
ax.plot(time_steps, prediction, label="einfache Vorhersage", linestyle="--")
ax.set_title("SimpleRNN-Grundidee: Zustand über die Zeit")
ax.set_xlabel("Zeitschritt")
ax.set_ylabel("Wert")
ax.legend()
plt.show()



### **2.3. LSTM oder GRU**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_02_03.jpg?v=1784816943" width="250">



>* LSTM und GRU speichern längere Kontexte
>* Kleine Modelle nutzen lernbares Gedächtnis effizient

>* LSTM: stärkeres Gedächtnis für lange Abhängigkeiten
>* GRU: kompakter, schneller, oft guter Start

>* Kleine Modelle sorgfältig zeitlich validieren
>* Nur bei klarer Fehlerverbesserung einsetzen



In [ ]:
#@title Python-Code - LSTM oder GRU

# Dieses Beispiel trainiert ein kleines GRU-Modell.
# Es zeigt Gedächtnis für kurze Sequenzen.
# Die Ausgabe vergleicht Vorhersage und Wahrheit.

import numpy as np
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_absolute_error
import sklearn

# Wir erzeugen eine glatte Zeitreihe mit Rauschen.
rng = np.random.default_rng(42)
time_steps = np.arange(260)
season = np.sin(time_steps * 2 * np.pi / 24)
trend = 0.003 * time_steps
noise = rng.normal(0, 0.08, size=time_steps.size)
series = season + trend + noise

# Sequenzfenster machen aus Zeitpunkten kleine Lernbeispiele.
window_size = 24
features = []
targets = []
for start in range(series.size - window_size):
    features.append(series[start:start + window_size])
    targets.append(series[start + window_size])

# Die Form prüft, ob jedes Beispiel ein Fenster enthält.
X = np.array(features)
y = np.array(targets)
if X.shape[1] != window_size:
    raise ValueError("Jedes Fenster muss genau 24 Werte enthalten.")

# Zeitreihen werden chronologisch getrennt, nicht zufällig gemischt.
split_index = int(0.75 * len(X))
X_train = X[:split_index]
X_test = X[split_index:]
y_train = y[:split_index]
y_test = y[split_index:]

# Diese kleine MLP simuliert hier ein kompaktes Sequenzmodell.
model = MLPRegressor(
    hidden_layer_sizes=(12,),
    activation="tanh",
    max_iter=500,
    random_state=42,
)
model.fit(X_train, y_train)

# Eine naive Baseline nutzt einfach den letzten Fensterwert.
baseline_pred = X_test[:, -1]
model_pred = model.predict(X_test)
baseline_mae = mean_absolute_error(y_test, baseline_pred)
model_mae = mean_absolute_error(y_test, model_pred)

print(f"scikit-learn-Version: {sklearn.__version__}")
print(f"Baseline MAE: {baseline_mae:.3f}")
print(f"Kleines Sequenzmodell MAE: {model_mae:.3f}")

# Die Grafik zeigt, ob das Modell den Verlauf glättet.
plt.figure(figsize=(8, 4))
plt.plot(y_test[:60], label="Wahrheit")
plt.plot(model_pred[:60], label="Vorhersage")
plt.title("Zeitreihenfenster: kleine sequenzielle Vorhersage")
plt.xlabel("Test-Zeitschritt")
plt.ylabel("Signalwert")
plt.legend()
plt.tight_layout()
plt.show()



## **3. Sequenzen fair bewerten**

### **3.1. Maskierte Sequenzen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_01.jpg?v=1784816929" width="250">



>* Masken trennen echte Werte von Platzhaltern
>* Nur gültige Zeitpunkte zählen zur Bewertung

>* Fehlende Messwerte fair aus Fehlern ausschließen
>* Modelle nur an gültigen Zeitpunkten vergleichen

>* Masken bewusst als Bewertungsdesign wählen
>* Fehler fachlich passend zusammenfassen



In [ ]:
#@title Python-Code - Maskierte Sequenzen

# Dieses Beispiel zeigt faire Fehlerbewertung mit Masken.
# Künstliche Auffüllwerte dürfen Kennzahlen nicht verzerren.
# Die Grafik vergleicht maskierte und unmaskierte Fehler.

import numpy as np
import matplotlib.pyplot as plt

# Drei Sequenzen haben unterschiedliche echte Längen.
true_sequences = [
    np.array([10.0, 12.0, 13.0, 15.0, 16.0, 18.0]),
    np.array([8.0, 9.0, 11.0, 12.0]),
    np.array([14.0, 13.0, 15.0]),
]

# Zwei einfache Modelle liefern Vorhersagen pro echter Sequenz.
classic_predictions = [
    np.array([10.5, 11.5, 12.5, 14.0, 15.0, 17.0]),
    np.array([8.5, 8.5, 10.0, 11.0]),
    np.array([13.5, 13.5, 14.0]),
]

network_predictions = [
    np.array([9.8, 12.2, 13.1, 15.2, 16.1, 18.2]),
    np.array([8.1, 9.2, 10.8, 12.1]),
    np.array([14.2, 12.8, 15.1]),
]

# Kürzere Sequenzen werden für Stapelverarbeitung aufgefüllt.
max_length = max(len(sequence) for sequence in true_sequences)
pad_value = 0.0

# Diese Hilfsfunktion erzeugt gepolsterte Werte und Masken.
def pad_with_mask(sequences, max_length, pad_value):
    padded = np.full((len(sequences), max_length), pad_value)
    mask = np.zeros((len(sequences), max_length), dtype=bool)

    for row, sequence in enumerate(sequences):
        length = len(sequence)
        padded[row, :length] = sequence
        mask[row, :length] = True

    return padded, mask

# Alle Daten erhalten dieselbe Maske gültiger Zeitpunkte.
y_true, valid_mask = pad_with_mask(true_sequences, max_length, pad_value)
y_classic, _ = pad_with_mask(classic_predictions, max_length, pad_value)
y_network, _ = pad_with_mask(network_predictions, max_length, pad_value)

# Eine einfache Prüfung schützt vor unpassenden Formen.
if y_true.shape != y_classic.shape or y_true.shape != y_network.shape:
    raise ValueError("Die Sequenzstapel müssen dieselbe Form haben.")

# Fehler werden einmal falsch und einmal fair berechnet.
classic_abs_error = np.abs(y_true - y_classic)
network_abs_error = np.abs(y_true - y_network)

classic_unmasked_mae = classic_abs_error.mean()
network_unmasked_mae = network_abs_error.mean()

classic_masked_mae = classic_abs_error[valid_mask].mean()
network_masked_mae = network_abs_error[valid_mask].mean()

# Zeitliche Fehlerprofile nutzen nur gültige Positionen.
valid_counts = valid_mask.sum(axis=0)
classic_time_mae = classic_abs_error.sum(axis=0) / valid_counts
network_time_mae = network_abs_error.sum(axis=0) / valid_counts

print("Unmaskierter MAE klassisch:", round(classic_unmasked_mae, 3))
print("Unmaskierter MAE Sequenznetz:", round(network_unmasked_mae, 3))
print("Maskierter MAE klassisch:", round(classic_masked_mae, 3))
print("Maskierter MAE Sequenznetz:", round(network_masked_mae, 3))
print("Gültige Zeitpunkte pro Spalte:", valid_counts.tolist())

# Die Grafik zeigt faire Fehler über die Zeit.
fig, ax = plt.subplots(figsize=(7, 4))
time_steps = np.arange(1, max_length + 1)

ax.plot(time_steps, classic_time_mae, marker="o", label="Klassisches Modell")
ax.plot(time_steps, network_time_mae, marker="o", label="Sequenznetz")
ax.set_title("Maskierte Fehleranalyse über Zeitpunkte")

ax.set_xlabel("Zeitpunkt im gepolsterten Fenster")
ax.set_ylabel("Maskierter mittlerer absoluter Fehler")
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()



### **3.2. Zeitliche Fehleranalyse**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_02.jpg?v=1784816931" width="250">



>* Fehlerzeitpunkte zeigen Modellstärken und Schwächen
>* Zeitverläufe prüfen praktische Einsatzfähigkeit

>* Fehler nach Sequenzabschnitten und Ereignissen betrachten
>* Muster, Verzögerungen und Fehlalarme erkennen

>* Fehler zeigen Ursachen von Modellunterschieden
>* Robustheit zählt über Übergänge hinweg



In [ ]:
#@title Python-Code - Zeitliche Fehleranalyse

# Wir vergleichen Fehler entlang einer Zeitreihe.
# Zwei einfache Modelle reagieren unterschiedlich schnell.
# Die Grafik zeigt zeitliche Fehler statt Mittelwerte.

import numpy as np
import matplotlib.pyplot as plt

# Eine kleine synthetische Zeitreihe enthält einen plötzlichen Sprung.
time_steps = np.arange(80)
base_signal = 20 + 0.05 * time_steps
seasonal_signal = 1.5 * np.sin(time_steps / 4)

# Der Sprung simuliert ein Ereignis, etwa geänderte Gebäudenutzung.
event_effect = np.where(time_steps >= 45, 6.0, 0.0)
true_values = base_signal + seasonal_signal + event_effect

# Modell A ist ein träger gleitender Mittelwert.
lagged_values = np.roll(true_values, 1)
lagged_values[0] = true_values[0]
smooth_model = 0.75 * lagged_values + 0.25 * true_values

# Modell B reagiert schneller, überschätzt aber kurz nach dem Ereignis.
fast_model = true_values.copy()
fast_model[45:52] = fast_model[45:52] + np.array([3, 2, 1, 0, -1, -1, 0])
fast_model[:45] = fast_model[:45] + 0.4 * np.sin(time_steps[:45] / 2)

# Absolute Fehler zeigen, wann ein Modell Probleme bekommt.
smooth_error = np.abs(true_values - smooth_model)
fast_error = np.abs(true_values - fast_model)

# Eine einfache Prüfung schützt vor unpassenden Arraylängen.
if len(true_values) != len(smooth_error) or len(true_values) != len(fast_error):
    raise ValueError("Die Zeitreihen müssen gleich lang sein.")

# Mittelwerte allein verdecken die zeitliche Struktur der Fehler.
print("Mittlerer Fehler gleitender Mittelwert:", round(float(smooth_error.mean()), 2))
print("Mittlerer Fehler schnelles Sequenzmodell:", round(float(fast_error.mean()), 2))
print("Größter Fehler nach Ereignis, gleitend:", round(float(smooth_error[45:55].max()), 2))
print("Größter Fehler nach Ereignis, schnell:", round(float(fast_error[45:55].max()), 2))

# Die Grafik macht Fehlerphasen entlang der Zeitachse sichtbar.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, smooth_error, label="Gleitender Mittelwert")
ax.plot(time_steps, fast_error, label="Schnelles Sequenzmodell")

# Die Ereignislinie markiert den kritischen Übergang.
ax.axvline(45, color="black", linestyle="--", label="Ereignis")
ax.set_title("Zeitliche Fehleranalyse zweier Vorhersagemodelle")
ax.set_xlabel("Zeitschritt")
ax.set_ylabel("Absoluter Fehler")

# Die Legende hilft beim fairen Modellvergleich.
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()



### **3.3. Sequenzen vergleichen**

<img src="https://cdn.jsdelivr.net/gh/mhrafiei/contents@main/Udemy/ML Für Anfänger/Module_19/Lecture_A/image_03_03.jpg?v=1784816927" width="250">



>* Nicht nur Gesamtfehler vergleichen
>* Stärken und Schwächen zeitlich prüfen

>* Vorhersagen können schnell oder verzögert reagieren
>* Fehler zeitlich statt nur punktweise analysieren

>* Gleiche Bedingungen sichern faire Modellvergleiche
>* Praktische Fehler entscheiden die Modellwahl



In [ ]:
#@title Python-Code - Sequenzen vergleichen

# Wir vergleichen Sequenzfehler über die Zeit.
# Ein glattes Modell reagiert oft verzögert.
# Die Grafik zeigt Fehler in Übergangsphasen.

import numpy as np
import matplotlib.pyplot as plt

# Diese künstliche Sequenz enthält ruhige und abrupte Abschnitte.
time_steps = np.arange(80)
true_signal = np.sin(time_steps / 8) + (time_steps >= 40) * 1.4

# Ein klassisches Signalmodell nutzt den letzten bekannten Wert.
classic_prediction = np.empty_like(true_signal)
classic_prediction[0] = true_signal[0]
classic_prediction[1:] = true_signal[:-1]

# Ein reaktiveres Sequenzmodell wird hier vereinfacht simuliert.
sequence_prediction = 0.75 * true_signal + 0.25 * classic_prediction

# Wir prüfen eine einfache Annahme zur Sequenzlänge.
if len(true_signal) != len(sequence_prediction):
    raise ValueError("Die Sequenzen müssen gleich lang sein.")

# Punktweise Fehler zeigen, wann ein Modell schwächelt.
classic_error = np.abs(true_signal - classic_prediction)
sequence_error = np.abs(true_signal - sequence_prediction)
transition_mask = (time_steps >= 38) & (time_steps <= 45)

# Mittelwerte allein verdecken oft wichtige zeitliche Unterschiede.
classic_mae = np.mean(classic_error)
sequence_mae = np.mean(sequence_error)
classic_transition_mae = np.mean(classic_error[transition_mask])
sequence_transition_mae = np.mean(sequence_error[transition_mask])

print(f"Klassisches Modell, MAE gesamt: {classic_mae:.3f}")
print(f"Sequenzmodell, MAE gesamt: {sequence_mae:.3f}")
print(f"Klassisches Modell, MAE Übergang: {classic_transition_mae:.3f}")
print(f"Sequenzmodell, MAE Übergang: {sequence_transition_mae:.3f}")

# Die Kurven machen die zeitliche Fehlerstruktur sichtbar.
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(time_steps, classic_error, label="klassisches Modell")
ax.plot(time_steps, sequence_error, label="Sequenzmodell")

ax.axvspan(38, 45, color="orange", alpha=0.2, label="Übergangsphase")
ax.set_title("Fehler über die Zeit statt nur Gesamtkennzahl")
ax.set_xlabel("Zeitpunkt")
ax.set_ylabel("absoluter Fehler")
ax.legend()

plt.show()



# <font color="#418FDE" size="6.5" uppercase>**Sequenzmodelle**</font>


In this lecture, you learned to:
- Bereiten Sequenzfenster und Baselines für zeitliche Vorhersage- oder Klassifikationsaufgaben vor. 
- Trainieren kleine Conv1D-, SimpleRNN- und LSTM- oder GRU-Modelle CPU-freundlich. 
- Vergleichen Sequenznetze mit klassischen Signalmodellen anhand zeitlicher Fehleranalysen. 

In the next Lecture (Lecture B), we will go over 'Autoencoder und Generierung'